In [ ]:
import os, cv2

path = os.path.join("/", "workspace", "data", "FFHQ", "ffhq256-images-only", "ffhq256", "40926.png")
cv2.imread(path)

In [1]:
import torch
from models.models import Model
from huggingface_hub import login
login(token="HF_TOKEN_REMOVED")
from models.attention_processor import StandAloneAttnProcessor as aspr
from diffusers.training_utils import compute_dream_and_update_latents


attn = aspr(hidden_size=768, scale=1.0)
model = Model()
# x = torch.randn(1, 3, 256, 256)
# context = torch.randn(1, 16, 768)
# out = model(x, x, ip_hidden_states=context)
# print(out.shape) 

/home/a6000/.local/share/virtualenvs/codes-npDqu6d0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/a6000/.local/share/virtualenvs/codes-npDqu6d0/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Number of Trainable Params: 99,840


In [2]:
x = torch.randn(1, 3, 256, 256)
x_sample = model.vae.encode(x).latent_dist.sample()
condition_vector = torch.randn(1, 16, 768)

In [3]:
model.unet

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_fe

In [4]:
latents = model.vae.encode(x).latent_dist.sample()
latents = latents * model.vae.config.scaling_factor
condition_vector = model.vae.encode(x).latent_dist.sample()
condition_vector = condition_vector * model.vae.config.scaling_factor
hidden_sates     = torch.randn(1, 16, 768)

# Sample Noise
noise = torch.randn_like(latents)
bsz = latents.shape[0]



timesteps = torch.randint(0, model.noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device)
timesteps = timesteps.long()
noisy_latents = model.noise_scheduler.add_noise(latents, noise, timesteps)


# Check mode
if model.noise_scheduler.config.prediction_type == "epsilon":
    target = noise
elif model.noise_scheduler.config.prediction_type == "v_prediction":
    target = model.noise_scheduler.get_velocity(latents, noise, timesteps)
else:
    raise ValueError(f"Unknown prediction type {model.noise_scheduler.config.prediction_type}")

#  Efficient Calculatiion but not use in v_prediction
if model.noise_scheduler.config.prediction_type == "epsilon":
    noisy_latents, target = compute_dream_and_update_latents(
        model.unet,
        model.noise_scheduler,
        timesteps,
        noise,
        noisy_latents,
        target,
        hidden_sates,
        1.0,
        )
# Predict the noise residual and compute loss
model_pred = model.unet(noisy_latents, timesteps, hidden_sates, return_dict=False,
                        cross_attention_kwargs={"ip_hidden_states": condition_vector})[0]


32
32
16
16
8
8
4
8
8
8
16
16
16
32
32
32


In [19]:
with open("./config/data/valid.txt", "r") as f:
    lines = f.readline().split(",")[:-1]
    print(len(lines))
    for line in lines:
        if "." not in line:
            print(line)

7000


In [20]:

lines[-1]

'55428.png'

In [5]:
model_pred.shape

torch.Size([1, 4, 32, 32])